# Movie Recommender System: Exploring sklearn's Non-Negative Matrix Factorization

This notebook explores the limitations of sklearn's Non-Negative Matrix Factorization (NMF) library for movie rating prediction.

In [4]:
import pandas as pd
import numpy as np
from sklearn.decomposition import NMF
from sklearn.metrics import mean_squared_error
import warnings
warnings.filterwarnings('ignore')

## Section 1: Matrix Factorization with sklearn NMF

In this section, we'll:
1. Load the movie ratings data
2. Create a user-item matrix
3. Apply sklearn's Non-Negative Matrix Factorization
4. Predict missing ratings
5. Calculate RMSE

### 1.1 Load Data

In [5]:
# Load all datasets
movies = pd.read_csv("data/movie-ratings/movies.csv")
training_data = pd.read_csv("data/movie-ratings/train.csv")
test_data = pd.read_csv("data/movie-ratings/test.csv")
users = pd.read_csv("data/movie-ratings/users.csv")

print(f"Training data shape: {training_data.shape}")
print(f"Test data shape: {test_data.shape}")
print(f"Number of movies: {len(movies)}")
print(f"Number of users: {len(users)}")
print("\nTraining data sample:")
print(training_data.head())
print("\nRating statistics:")
print(training_data['rating'].describe())

Training data shape: (700146, 3)
Test data shape: (300063, 3)
Number of movies: 3883
Number of users: 6040

Training data sample:
    uID   mID  rating
0   744  1210       5
1  3040  1584       4
2  1451  1293       5
3  5455  3176       2
4  2507  3074       5

Rating statistics:
count    700146.000000
mean          3.581589
std           1.117508
min           1.000000
25%           3.000000
50%           4.000000
75%           4.000000
max           5.000000
Name: rating, dtype: float64


### 1.2 Create User-Item Matrix

In [6]:
# Create user-item matrix from training data
# Pivot the data to create a matrix where rows are users and columns are movies
user_item_matrix = training_data.pivot_table(
    index='uID', 
    columns='mID', 
    values='rating'
)

print(f"User-Item Matrix shape: {user_item_matrix.shape}")
print(f"Sparsity: {(user_item_matrix.isna().sum().sum() / (user_item_matrix.shape[0] * user_item_matrix.shape[1])) * 100:.2f}%")

# NMF requires non-negative values and no missing values
# We'll fill missing values with 0 (this is a key limitation we'll discuss)
user_item_matrix_filled = user_item_matrix.fillna(0)

print(f"\nMatrix after filling NaN with 0:")
print(f"Shape: {user_item_matrix_filled.shape}")
print(f"Min value: {user_item_matrix_filled.min().min()}")
print(f"Max value: {user_item_matrix_filled.max().max()}")

User-Item Matrix shape: (6040, 3664)
Sparsity: 96.84%

Matrix after filling NaN with 0:
Shape: (6040, 3664)
Min value: 0.0
Max value: 5.0


### 1.3 Apply sklearn's Non-Negative Matrix Factorization

In [7]:
# Apply NMF with different numbers of components
n_components = 20  # Number of latent features

print(f"Applying NMF with {n_components} components...")

# Initialize and fit NMF model
nmf_model = NMF(
    n_components=n_components,
    init='random',
    random_state=42,
    max_iter=200,
    alpha_W=0.0,
    alpha_H=0.0
)

# Fit the model and transform the data
# W: User features (users x components)
# H: Item features (components x items)
W = nmf_model.fit_transform(user_item_matrix_filled)
H = nmf_model.components_

print(f"User feature matrix (W) shape: {W.shape}")
print(f"Item feature matrix (H) shape: {H.shape}")
print(f"Reconstruction error: {nmf_model.reconstruction_err_:.4f}")
print(f"Number of iterations: {nmf_model.n_iter_}")

Applying NMF with 20 components...
User feature matrix (W) shape: (6040, 20)
Item feature matrix (H) shape: (20, 3664)
Reconstruction error: 2633.4808
Number of iterations: 200


### 1.4 Reconstruct Ratings Matrix

In [8]:
# Reconstruct the full ratings matrix
predicted_ratings = np.dot(W, H)

# Convert back to DataFrame with proper indices
predicted_ratings_df = pd.DataFrame(
    predicted_ratings,
    index=user_item_matrix_filled.index,
    columns=user_item_matrix_filled.columns
)

print(f"Predicted ratings shape: {predicted_ratings_df.shape}")
print(f"\nPredicted ratings statistics:")
print(f"Min: {predicted_ratings_df.min().min():.4f}")
print(f"Max: {predicted_ratings_df.max().max():.4f}")
print(f"Mean: {predicted_ratings_df.mean().mean():.4f}")
print(f"Std: {predicted_ratings_df.std().mean():.4f}")

# Check sample predictions
print("\nSample predicted ratings:")
print(predicted_ratings_df.iloc[:5, :5])

Predicted ratings shape: (6040, 3664)

Predicted ratings statistics:
Min: 0.0000
Max: 7.3214
Mean: 0.1311
Std: 0.1861

Sample predicted ratings:
mID         1         2         3         4         5
uID                                                  
1    1.709725  0.549405  0.014191  0.023687  0.015364
2    1.594666  0.335222  0.099675  0.035014  0.059429
3    0.840486  0.146931  0.014632  0.000931  0.000000
4    0.000000  0.000066  0.003995  0.000000  0.000000
5    0.658992  0.038826  0.013240  0.073820  0.000000


### 1.5 Predict Test Set Ratings and Calculate RMSE

In [9]:
# Extract predictions for test set
test_predictions = []
test_actuals = []
missing_count = 0

for idx, row in test_data.iterrows():
    user_id = row['uID']
    movie_id = row['mID']
    actual_rating = row['rating']
    
    # Check if user and movie exist in our training matrix
    if user_id in predicted_ratings_df.index and movie_id in predicted_ratings_df.columns:
        predicted_rating = predicted_ratings_df.loc[user_id, movie_id]
        # Clip predictions to valid rating range (0.5 to 5.0)
        predicted_rating = np.clip(predicted_rating, 0.5, 5.0)
        test_predictions.append(predicted_rating)
        test_actuals.append(actual_rating)
    else:
        missing_count += 1

print(f"Test cases with predictions: {len(test_predictions)}")
print(f"Test cases missing (user or movie not in training): {missing_count}")
print(f"Coverage: {len(test_predictions) / len(test_data) * 100:.2f}%")

Test cases with predictions: 300006
Test cases missing (user or movie not in training): 57
Coverage: 99.98%


In [10]:
# Calculate RMSE
rmse = np.sqrt(mean_squared_error(test_actuals, test_predictions))

print(f"\n{'='*60}")
print(f"RMSE on Test Set: {rmse:.4f}")
print(f"{'='*60}")

# Additional metrics for context
mae = np.mean(np.abs(np.array(test_actuals) - np.array(test_predictions)))
print(f"\nMean Absolute Error (MAE): {mae:.4f}")
print(f"\nActual ratings - Mean: {np.mean(test_actuals):.4f}, Std: {np.std(test_actuals):.4f}")
print(f"Predicted ratings - Mean: {np.mean(test_predictions):.4f}, Std: {np.std(test_predictions):.4f}")

# Show distribution of errors
errors = np.array(test_actuals) - np.array(test_predictions)
print(f"\nError distribution:")
print(f"Mean error: {np.mean(errors):.4f}")
print(f"Std of errors: {np.std(errors):.4f}")
print(f"Min error: {np.min(errors):.4f}")
print(f"Max error: {np.max(errors):.4f}")


RMSE on Test Set: 2.7638

Mean Absolute Error (MAE): 2.5206

Actual ratings - Mean: 3.5816, Std: 1.1161
Predicted ratings - Mean: 1.0753, Std: 0.7373

Error distribution:
Mean error: 2.5063
Std of errors: 1.1647
Min error: -3.6775
Max error: 4.5000


### 1.6 Compare with Simple Baseline

In [11]:
# Let's compare with a simple baseline: global mean
global_mean = training_data['rating'].mean()
baseline_predictions = [global_mean] * len(test_actuals)
baseline_rmse = np.sqrt(mean_squared_error(test_actuals, baseline_predictions))

print(f"Global mean baseline: {global_mean:.4f}")
print(f"Baseline RMSE: {baseline_rmse:.4f}")
print(f"\nNMF RMSE: {rmse:.4f}")
print(f"Improvement over baseline: {((baseline_rmse - rmse) / baseline_rmse * 100):.2f}%")

# User mean baseline
user_means = training_data.groupby('uID')['rating'].mean()
user_baseline_predictions = []
for idx, row in test_data.iterrows():
    user_id = row['uID']
    movie_id = row['mID']
    if user_id in predicted_ratings_df.index and movie_id in predicted_ratings_df.columns:
        if user_id in user_means.index:
            user_baseline_predictions.append(user_means[user_id])
        else:
            user_baseline_predictions.append(global_mean)

user_baseline_rmse = np.sqrt(mean_squared_error(test_actuals, user_baseline_predictions))
print(f"\nUser mean baseline RMSE: {user_baseline_rmse:.4f}")
print(f"Improvement over user baseline: {((user_baseline_rmse - rmse) / user_baseline_rmse * 100):.2f}%")

Global mean baseline: 3.5816
Baseline RMSE: 1.1161

NMF RMSE: 2.7638
Improvement over baseline: -147.63%

User mean baseline RMSE: 1.0353
Improvement over user baseline: -166.96%


## Section 2: Analysis and Discussion

### Why sklearn's NMF Didn't Work Well

### 2.1 Key Limitations of sklearn's NMF for Collaborative Filtering

Based on the results above, we can identify several critical limitations:

#### **1. Missing Value Problem**

- **The Issue**: sklearn's NMF requires a complete matrix (no NaN values)
- **Our Workaround**: We filled missing values with 0
- **Why This is Problematic**:
  - Rating matrices are highly sparse (often 90%+ missing values)
  - Filling with 0 treats "unrated" the same as "rated very poorly"
  - NMF tries to reconstruct these 0s, which distorts the model
  - The algorithm optimizes for these artificial 0s rather than actual ratings

#### **2. Non-Negativity Constraint**

- **The Issue**: NMF requires all values to be ≥ 0
- **Why This is Problematic**:
  - Collaborative filtering often benefits from mean-centering (subtracting user/item means)
  - Mean-centered data can have negative values
  - Cannot model user biases effectively (some users rate higher/lower on average)
  - Cannot capture deviations from expected ratings

#### **3. Wrong Optimization Objective**

- **The Issue**: NMF minimizes reconstruction error on ALL matrix entries
- **Why This is Problematic**:
  - We only care about predicting OBSERVED ratings accurately
  - The model wastes capacity trying to "reconstruct" the filled 0s
  - This leads to overfitting on the artificial structure we created

#### **4. Lack of Regularization for Collaborative Filtering**

- sklearn's NMF has basic L1/L2 regularization (alpha_W, alpha_H)
- But it doesn't have:
  - User/item bias terms
  - Proper handling of implicit feedback
  - Confidence weighting for different observations

#### **5. Performance Comparison**

As we saw above:
- NMF may perform poorly or only marginally better than simple baselines
- Simple user-mean baseline can sometimes outperform NMF
- Similarity-based methods (user-user, item-item) typically perform better because:
  - They only use observed ratings
  - They naturally handle sparsity
  - They don't require filling missing values

### 2.2 Suggested Fixes and Alternatives

#### **Option 1: Use Specialized Libraries for Matrix Factorization**

Instead of sklearn's NMF, use libraries designed for collaborative filtering:

1. **Surprise library**:
   ```python
   from surprise import SVD, Dataset, Reader
   # SVD handles sparse data natively
   # Only trains on observed ratings
   # Includes user/item biases
   ```

2. **Implicit library** (for implicit feedback):
   ```python
   from implicit.als import AlternatingLeastSquares
   # Designed for sparse matrices
   # Uses confidence weighting
   ```

3. **LightFM**:
   - Combines collaborative and content-based filtering
   - Handles cold-start problems

#### **Option 2: Modify the NMF Approach**

If you must use sklearn's NMF, try these improvements:

1. **Better Missing Value Handling**:
   ```python
   # Instead of filling with 0, use user/item means
   user_means = training_data.groupby('uID')['rating'].mean()
   item_means = training_data.groupby('mID')['rating'].mean()
   # Fill with a combination of both
   ```

2. **Iterative Imputation**:
   ```python
   from sklearn.impute import IterativeImputer
   # Use multiple iterations to estimate missing values
   # Then apply NMF
   ```

3. **Weighted NMF** (custom implementation):
   - Create a weight matrix (1 for observed, 0 for missing)
   - Modify the loss function to only consider observed ratings
   - This requires implementing custom NMF

#### **Recommended Solution**

For movie recommendations:

1. **Use the Surprise library's SVD or SVD++**:
   - Specifically designed for rating prediction
   - Handles sparsity naturally
   - Includes bias terms
   - Easy to use and well-documented

2. **Or implement ALS (Alternating Least Squares)**:
   - Popular in industry (used by Spotify, Netflix)
   - Handles implicit feedback well
   - Scalable to large datasets

3. **For the best results, use ensemble methods**:
   - Combine matrix factorization with neighborhood methods
   - Add content-based features when available
   - Use neural collaborative filtering for complex patterns

### 2.4 Conclusion

**Key Takeaways**:

1. **sklearn's NMF is not designed for collaborative filtering** - it's meant for dimensionality reduction and topic modeling on complete matrices

2. **The main problem is handling missing data** - filling with 0 fundamentally breaks the collaborative filtering assumption

3. **Use specialized libraries** - Surprise, Implicit, or LightFM are better choices for recommendation systems

4. **When NMF works well**:
   - Dense matrices (few missing values)
   - Non-negative data that's meaningful (e.g., word counts, image pixels)
   - Topic modeling and feature extraction

5. **For movie recommendations, prefer**:
   - SVD with bias terms (Surprise library)
   - ALS (Implicit library)
   - Neural collaborative filtering (PyTorch/TensorFlow)
   - Ensemble methods combining multiple approaches

The poor performance of sklearn's NMF on this task is not a flaw in the algorithm itself, but rather using the wrong tool for the job. Matrix factorization can work very well for collaborative filtering, but it needs to be specifically designed to handle sparse observed ratings rather than trying to reconstruct a filled-in dense matrix.